# kathareel — Hinglish explainer demo (Kaggle)

**kathareel** ek skill hai jo kisi bhi topic ka motion-graphics explainer video banata hai:
**voiceover Hinglish** (Hindi bola hua, Latin script me likha) + **board par har likha hua word English**.
Har frame [Remotion](https://remotion.dev) (React + TypeScript) se code me draw hota hai — koi stock footage nahi.

Ye notebook ek **chhota 37-second demo** end-to-end banata hai, taaki tum khud dekh sako ki sab chal raha hai:

| Step | Kya hota hai |
|---|---|
| 1 | repo clone + system deps (ffmpeg, espeak-ng, Chromium libs) |
| 2 | demo project setup (template + demo narration + demo shots) |
| 3 | `tts_build.py` — Hinglish voiceover + frame-accurate subtitles |
| 4 | `remotion render` — 1280x720 H.264 MP4 |
| 5 | contact sheet + inline video player |

> **Note:** Kaggle se Microsoft ka edge-tts endpoint block hota hai, isliye yahan **espeak-ng (offline Hindi voice)** use hoti hai — aawaz robotic hai, par ye proof hai ki poori pipeline chalti hai. Apne laptop par wahi script `edge-tts hi-IN-MadhurNeural` (natural) use karti hai.


## 1. Setup — repo clone + system packages

`Internet` on hona chahiye (Notebook settings → Internet → On).


In [ ]:
import subprocess, sys, os

# system packages: ffmpeg (audio/video), espeak-ng (offline Hindi TTS),
# aur Chromium ke shared libs jo Remotion ko chahiye
subprocess.run('apt-get -qq update', shell=True, check=False)
subprocess.run('apt-get -qq install -y ffmpeg espeak-ng libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 '
               'libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 '
               'libgbm1 libasound2', shell=True, check=False)
print(subprocess.run('ffmpeg -version | head -1; espeak-ng --version | head -1', shell=True,
                     capture_output=True, text=True).stdout)


In [ ]:
REPO_URL = 'https://github.com/amaniimtuniversity-ux/ea_auto.git'
WORK = '/kaggle/working'

if not os.path.isdir(f'{WORK}/repo/.git'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, f'{WORK}/repo'], check=True)
else:
    subprocess.run(['git', '-C', f'{WORK}/repo', 'pull'], check=False)
print(os.listdir(f'{WORK}/repo'))


## 2. Demo project setup

`setup_demo.py` template se project banata hai, `lang: 'hi'` (Hinglish voice + English board) set karta hai,
`npm install` chalata hai aur TTS karta hai. Yahan `--skip-render` diya hai taaki render alag cell me dikhe.


In [ ]:
!python {WORK}/repo/kaggle/setup_demo.py --work {WORK}/demo --skip-render


## 3. Verify karo ki sahi laga — config + narration

Yahan do cheezein dekh lo: `lang: 'hi'` (Hinglish voice) aur `builtBy: 'built by raghv'`.


In [ ]:
print(open(f'{WORK}/demo/src/config.ts').read()[:1800])
print('------------- narration.txt -------------')
print(open(f'{WORK}/demo/script/narration.txt').read())


## 4. Voiceover + timeline

`tts_build.py` narration ko sentence-by-sentence synthesize karta hai aur `timeline.ts` + `subs.ts` likhta hai
(subtitle timing frames me, audio ke word boundaries se). Ye cell dikhata hai ki kya bana.


In [ ]:
import json
tl = json.load(open(f'{WORK}/demo/script/timeline.json'))
print('engine :', tl['engine'], '| voice :', tl['voice'])
print('frames :', tl['total_frames'], '=', round(tl['total_frames']/tl['fps'], 1), 'sec')
print('lang   :', tl['lang'], '(board)', '| voice lang:', tl.get('voice_lang'))
for s in tl['sentences']:
    print(f"  {s['id']} f{s['from']}-{s['to']}  {s['text'][:70]}")


## 5. Render the video

2 CPU cores par ~1.5 minute lagta hai. `--concurrency=2` Kaggle ke liye theek hai.


In [ ]:
!cd {WORK}/demo && npx remotion render src/index.ts Demo demo.mp4 --concurrency=2
!ls -lh {WORK}/demo/demo.mp4


## 6. Watch it


In [ ]:
from IPython.display import Video, display
display(Video(f'{WORK}/demo/demo.mp4', embed=True, width=960))


## 7. Contact sheet — 6 frames ek nazar me

Yahan check kar sakte ho: board par sab English hai, subtitles Hinglish hain, HUD aur progress bar English.


In [ ]:
!ffmpeg -v error -y -i {WORK}/demo/demo.mp4 -vf "fps=1/6,scale=640:-1,tile=3x2" -frames:v 1 {WORK}/demo/contact_sheet.png
from IPython.display import Image, display
display(Image(f'{WORK}/demo/contact_sheet.png'))


## 8. Apna topic chalao (optional)

Yahan se aage asli skill shuru hoti hai: `template/` copy karke apni narration likho,
`config.ts` me title/chapters/HUD bharo, aur phir agent ko `SKILL.md` ke 9 stages chalane do.

```python
# naya project
!python {WORK}/repo/template/scripts/new_project.sh {WORK}/myvideo mytopic

# narration.txt likho (Hinglish, har line me | se subtitle blocks),
# config.ts me lang: 'hi', title, chapters, hud bharo, phir:
!cd {WORK}/myvideo && python3 scripts/tts_build.py
```

Board rule yaad rakho: **jo screen par likha hai wo English hai** — sirf aawaz Hinglish hai.
End credit: `built by raghv`.
